 Sistema de Gestión de Vehículos Autónomos
Diseñar la arquitectura de un VehiculoAutonomo compuesto por un módulo de navegación (SistemaGPS), un conjunto de sensores de entorno (LiDAR, CamaraVision, SensorUltrasonico), un controlador de propulsión (UnidadMotor) y un motor de toma de decisiones (IAConduccion), permitiendo evaluar la ruta y reaccionar ante obstáculos combinando sus componentes.



In [ ]:
class SistemaGPS:
    def __init__(self):
        self.destino = None
        self.tramos_restantes = 0

    def establecer_destino(self, destino, tramos=5):
        self.destino = destino
        self.tramos_restantes = tramos

    def avanzar_tramo(self):
        self.tramos_restantes -= 1

    def llego_a_destino(self):
        return self.tramos_restantes == 0


class Sensor:
    def __init__(self, nombre, alcance_max):
        self.nombre = nombre
        self.alcance_max = alcance_max  # metros

    def detectar_obstaculo(self, distancia):
        # Devuelve la distancia si el obstáculo está al alcance, si no None
        if distancia is not None and distancia <= self.alcance_max:
            return distancia
        return None


class LiDAR(Sensor):
    def __init__(self):
        super().__init__("LiDAR", 100)


class CamaraVision(Sensor):
    def __init__(self):
        super().__init__("CamaraVision", 50)


class SensorUltrasonico(Sensor):
    def __init__(self):
        super().__init__("SensorUltrasonico", 5)


class UnidadMotor:
    def __init__(self):
        self.velocidad = 0
        self.direccion = "recto"

    def acelerar(self):
        self.velocidad += 10
        self.direccion = "recto"

    def reducir(self):
        self.velocidad = max(0, self.velocidad - 10)

    def frenar(self):
        self.velocidad = 0

    def girar(self, direccion):
        self.direccion = direccion


class IAConduccion:
    def decidir(self, lecturas):
        if not lecturas:
            return "avanzar"
        mas_cercano = min(lecturas)
        if mas_cercano <= 5:
            return "frenar"
        if mas_cercano <= 20:
            return "esquivar"
        return "reducir"


class VehiculoAutonomo:
    def __init__(self):
        self.gps = SistemaGPS()
        self.sensores = [LiDAR(), CamaraVision(), SensorUltrasonico()]
        self.motor = UnidadMotor()
        self.ia = IAConduccion()

    def evaluar_ruta(self, distancia):
        # 1. Los sensores leen el entorno
        lecturas = []
        for sensor in self.sensores:
            medida = sensor.detectar_obstaculo(distancia)
            if medida is not None:
                lecturas.append(medida)

        # 2. Como se especifica, la ia toma la decision
        decision = self.ia.decidir(lecturas)

        # 3. Se determinan las acciones del motor
        if decision == "avanzar":
            self.motor.acelerar()
        elif decision == "reducir":
            self.motor.reducir()
        elif decision == "esquivar":
            self.motor.reducir()
            self.motor.girar("izquierda")
        elif decision == "frenar":
            self.motor.frenar()

        print(f"Obstáculo: {distancia} m -> {decision} | "
              f"velocidad: {self.motor.velocidad} km/h | dirección: {self.motor.direccion}")
        return decision

    def conducir(self, destino, obstaculos):
        self.gps.establecer_destino(destino)
        print(f"Viaje hacia {destino}\n")

        ciclo = 0
        while not self.gps.llego_a_destino():
            distancia = obstaculos[ciclo] if ciclo < len(obstaculos) else None
            ciclo += 1
            decision = self.evaluar_ruta(distancia)
            if decision != "frenar":
                self.gps.avanzar_tramo()

        self.motor.frenar()
        print(f"\nLlegó a {destino}")


auto = VehiculoAutonomo()
auto.conducir("Castro", [None, 60, 15, 3, None, None])

Viaje hacia Castro

Obstáculo: None m -> avanzar | velocidad: 10 km/h | dirección: recto
Obstáculo: 60 m -> reducir | velocidad: 0 km/h | dirección: recto
Obstáculo: 15 m -> esquivar | velocidad: 0 km/h | dirección: izquierda
Obstáculo: 3 m -> frenar | velocidad: 0 km/h | dirección: izquierda
Obstáculo: None m -> avanzar | velocidad: 10 km/h | dirección: recto
Obstáculo: None m -> avanzar | velocidad: 20 km/h | dirección: recto

Llegó a Castro
